# 02 · Exploring the MBTA API for one bus route
In this notebook I look at the MBTA API for one bus route that is also in my files. Route `01` in my
files is called `1` in the API. I want to find out which API fields match my file columns, which
column the API doesn't have, and how many records one route gives me for one day.

In [1]:
import os
from datetime import date

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
BASE = "https://api-v3.mbta.com"
HEADERS = {"x-api-key": os.environ["MBTA_API_KEY"]} if os.environ.get("MBTA_API_KEY") else {}
TODAY = date.today().isoformat()

## Routes: `GET /routes/1` and its `direction_names`

In [2]:
r = requests.get(f"{BASE}/routes/1", headers=HEADERS)
print(r.status_code)
attrs = r.json()["data"]["attributes"]
attrs["short_name"], attrs["direction_names"]

200


('1', ['Outbound', 'Inbound'])

**What `direction_names` shows:**

- The list is `['Outbound', 'Inbound']`, so spot 0 is Outbound and spot 1 is Inbound
- That means `direction_id` 1 = Inbound and 0 = Outbound, which are the same words my files use

## Schedules: `GET /schedules?filter[route]=1&filter[date]=<today>`, one record

In [3]:
r = requests.get(f"{BASE}/schedules", params={"filter[route]": "1", "filter[date]": TODAY}, headers=HEADERS)
print(r.status_code, "records:", len(r.json()["data"]))
r.json()["data"][0]

200 records: 5160


{'attributes': {'arrival_time': None,
  'departure_time': '2026-10-14T04:37:00-04:00',
  'direction_id': 0,
  'drop_off_type': 1,
  'pickup_type': 0,
  'stop_headsign': None,
  'stop_sequence': 1,
  'timepoint': True},
 'id': 'schedule-78181030-64-1',
 'relationships': {'added_routes': {'data': []},
  'route': {'data': {'id': '1', 'type': 'route'}},
  'stop': {'data': {'id': '64', 'type': 'stop'}},
  'trip': {'data': {'id': '78181030', 'type': 'trip'}}},
 'type': 'schedule'}

In [4]:
data = r.json()["data"]
sched = pd.DataFrame([{
    "direction_id": d["attributes"]["direction_id"],
    "stop_id": d["relationships"]["stop"]["data"]["id"],
    "trip_id": d["relationships"]["trip"]["data"]["id"],
    "arrival_time": d["attributes"]["arrival_time"],
    "departure_time": d["attributes"]["departure_time"],
    "timepoint": d["attributes"].get("timepoint"),
} for d in data])
print("timepoint rows:", sched["timepoint"].sum(), "of", len(sched))
sched.head()

timepoint rows: 215 of 5160


,direction_id,stop_id,trip_id,arrival_time,departure_time,timepoint
0,0,64,78181030,None,2026-10-14T04:37:00-04:00,True
1,0,1,78181030,2026-10-14T04:37:00-04:00,2026-10-14T04:37:00-04:00,False
2,0,2,78181030,2026-10-14T04:38:00-04:00,2026-10-14T04:38:00-04:00,False
3,0,6,78181030,2026-10-14T04:39:00-04:00,2026-10-14T04:39:00-04:00,False
4,0,10003,78181030,2026-10-14T04:40:00-04:00,2026-10-14T04:40:00-04:00,False


## Predictions: the live guesses for the same route

In [5]:
r = requests.get(f"{BASE}/predictions", params={"filter[route]": "1", "page[limit]": "3"}, headers=HEADERS)
print(r.status_code)
[d["attributes"] for d in r.json()["data"]]

200


[{'arrival_time': None,
  'arrival_uncertainty': None,
  'departure_time': '2026-10-15T00:07:00-04:00',
  'departure_uncertainty': None,
  'direction_id': 1,
  'last_trip': False,
  'revenue': 'REVENUE',
  'schedule_relationship': None,
  'status': None,
  'stop_sequence': 1,
  'trip_headsign': None,
  'update_type': None},
 {'arrival_time': None,
  'arrival_uncertainty': None,
  'departure_time': '2026-10-15T00:50:00-04:00',
  'departure_uncertainty': None,
  'direction_id': 1,
  'last_trip': False,
  'revenue': 'REVENUE',
  'schedule_relationship': None,
  'status': None,
  'stop_sequence': 1,
  'trip_headsign': None,
  'update_type': None},
 {'arrival_time': None,
  'arrival_uncertainty': None,
  'departure_time': '2026-10-15T00:00:00-04:00',
  'departure_uncertainty': None,
  'direction_id': 0,
  'last_trip': False,
  'revenue': 'REVENUE',
  'schedule_relationship': None,
  'status': None,
  'stop_sequence': 1,
  'trip_headsign': None,
  'update_type': None}]

## What I found

**Which API field gives each file column:**
- `service_date` → the date I asked for in `filter[date]`
- `route_id` → `relationships.route.data.id`
- `direction_id` → `attributes.direction_id` (1 = Inbound, 0 = Outbound)
- `stop_id` → `relationships.stop.data.id`
- `half_trip_id` → `relationships.trip.data.id`
- `scheduled` → `attributes.departure_time` (or `arrival_time` at the last stop)

**Which file columns the API doesn't have:**
- `actual`, because a schedule is only the plan, not what really happened
- also `time_point_id`, `time_point_order`, `point_type`, `standard_type`, `scheduled_headway` and `headway`

**How many records one route gives for one day:**
- 5160 records for route 1 today (every stop on every trip)